In [1]:
#I will now attempt to create a script that will let you generate an NxN matrix and choose tiles to blot out. 
#This will likely be done by typing in which tiles need to be blotted out. Start by manually blotting out all the tiles
#Keep track of blotted and not blotted with a 2D matrix.
#Lastly, I would like this code to automatically number the cells (which is the most annoying part to do by hand)

#READ ME :D
#You're going to need to install Pillow for python
#You're also going to need to install some freeeee fonts: https://online-fonts.com/fonts/freemono

#Completed goal: make grid
#Completed goal: Turn selected squares black (select by filling in array sadly :())
#Next goal: add a number to the center to each cell. We just want to show that we can put a number in a cell. 

from PIL import Image
from PIL import ImageDraw
from PIL import ImageFont
import numpy as np

#Right now you need to manually enter the boxes that are blacked out. Also right now don't do anything NxM N!=M now... Just squares

#Kaleigh's crossword below
bandw = [[0,0,0,1,0,0,0,0,0,0,0,0,0,0,1],
         [0,0,0,1,0,0,0,0,0,1,1,0,0,0,0],
         [0,0,0,0,0,0,0,0,0,0,1,0,0,0,0],
         [0,0,0,0,0,0,1,0,0,0,1,1,0,0,0],
         [0,0,0,0,1,1,0,0,0,0,0,0,0,1,1],
         [1,1,1,0,1,1,0,0,1,0,0,1,0,0,0],
         [0,0,0,0,0,0,0,0,0,0,0,0,1,0,0],
         [0,0,0,1,0,0,0,1,0,0,0,1,0,0,0],
         [0,0,1,0,0,0,0,0,0,0,0,0,0,0,0],
         [0,0,0,1,0,0,1,0,0,1,1,0,1,1,1],
         [1,1,0,0,0,0,0,0,0,1,1,0,0,0,0],
         [0,0,0,1,1,0,0,0,1,0,0,0,0,0,0],
         [0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
         [0,0,0,0,1,1,0,0,0,0,0,1,0,0,0],
         [1,0,0,0,0,0,0,0,0,0,0,1,0,0,0]
        ]

#pixels for now, but I want real rows and columns later
size = len(bandw) #NxN for crossword, use the inputted matrix to make the crossword
thk = 5 #Thickness in pixels of each line
numpix = 2000 #approximate pixel size of the crossword
fontsize = 30
cwname = 'TestCW.png' #Name the crossword here

grid_spacing = int(numpix/size) #The space between each line, calculated from the approximate number of pixels
actpix = size*grid_spacing + thk #Actual pixel size that has size number of boxes with a line of thickness thk on the bottom
blkbox = grid_spacing-thk #This is the exact size of the pixel box needed to color in a white square to black

#Color of gridlines is grayscale, 0 is black and 255 is white

pixels = np.zeros((actpix,actpix,3),dtype=np.uint8)
pixels[:,:] = (255,255,255) #Color all pixels white, this is the space where changing colors is good!

#Identify the location of the upper left corner pixel of white so we can color black over it
#thk,thk for (1,1) 138, thk+grid_spacing,thk+grid_spacing for (2,2) you get the picture

row = 0 #Track the row number
col = 0 #Track the column number
lc = 0 #Lines will be black
line_color = (lc,lc,lc) #tuple made of lc. 
axis_color = (0,0,0) #Makes it black. I think I want that.

for r in bandw: #For each row in bandw is how that goes down
    for c in r: #For each element in each row
        if c != 0: #Check the element c to see if the square should be black and color it in if it is
            pixels[row*grid_spacing+thk:row*grid_spacing+thk+blkbox,col*grid_spacing+thk:col*grid_spacing+thk+blkbox] = line_color
        col = col + 1 
    row = row + 1
    col = 0 

for row in range(0,actpix,grid_spacing): #Add row lines based on the grid_spacing value
    for width in range(thk):
        pixels[row+width,:] = line_color

for col in range(0,actpix,grid_spacing): #Add col lines based on the grid_spacing value
    for width in range(thk):
        pixels[:,col+width,] = line_color
        
img = Image.fromarray(pixels) #Now we create an image from the pixel map
img.save('crossword.png') #Save a temporary png with no numbers yet

#This isn't the most efficient way to write the numbers but I will do it like this for now. New loops for numbers.
#This is for when I'm ready to put numbers into the crossword
#If I was better at coding I'd combine the below section with the above one. This does have the beneft(?) of 
#creating a crossword.png of the non-numbered version though. 

#You do need the FreeMonoBold ttf file, but you can set the fontsize with that. It's in the same folder as this notebook
myFont = ImageFont.truetype('/home/kelsey/Downloads/FreeMono/FreeMonospacedBold.ttf', fontsize)
I1 = ImageDraw.Draw(img) #We open the file with the intention of drawing (letters) on top of it

# Add Text to an image
row = 0 #For iterating rows
col = 0 #For iterating columns
num = 1 #This starts at 1 because crosswords don't start at 0 lol
filled = 0 #Track whether or not in a loop a number has already been asigned so you don't number over twice

for r in bandw: #For each row in bandw
    for c in r: #For each element in each row
        if c != 1: #If the square is black then do not number it
            #Start with across
            if filled == 0 and col == 0: #If we're not filled yet and we are on the column 0, number it
                if bandw[row][col+1] == 0: #You also need to check that the square to the right is not black
                    I1.text((3+col*grid_spacing+thk,row*grid_spacing+thk), str(num), font=myFont, fill=(0, 0, 0))
                    num = num + 1 #If you do number it, then increase number
                    filled = 1 #And also flag it as filled
            if filled == 0 and col != size-1: #If not filled and we are not at the edge (size-1) of the crossword
                if bandw[row][col-1] != 0 and bandw[row][col+1] == 0: #Check if left is black and right is white
                    I1.text((3+col*grid_spacing+thk,row*grid_spacing+thk), str(num), font=myFont, fill=(0, 0, 0))
                    num = num + 1 
                    filled = 1
            if filled == 0 and row == 0: #If we're not filled yet and we are on the row 0, number it
                if bandw[row+1][col] == 0: #You also need to check that the square one under is not black
                    I1.text((3+col*grid_spacing+thk,row*grid_spacing+thk), str(num), font=myFont, fill=(0, 0, 0))
                    num = num + 1
                    filled = 1
            if filled == 0 and row != size-1: #If not filled and we are not at the bottom of the (size-1) crossword
                if bandw[row-1][col] != 0 and bandw[row+1][col] == 0: #Check that above is black and below is white
                    I1.text((3+col*grid_spacing+thk,row*grid_spacing+thk), str(num), font=myFont, fill=(0, 0, 0))
                    num = num + 1
                    filled = 1
        col = col + 1 #Iterate columns
        filled = 0 #Reset the flag
    row = row + 1 #Iterate rows
    col = 0 #Reset column number because it's a new row

img.save(cwname) #Save the new art with the name.


#Debugging tool, you can output the row and column that triggers an if statement
        #print("(",end="") #The end="" makes is so there's no new line automatically
        #print(row,end="") #The end="" makes is so there's no new line automatically
        #print(",",end="") #The end="" makes is so there's no new line automatically
        #print(col,end="") #The end="" makes is so there's no new line automatically
        #print(")",end="") #The end="" makes is so there's no new line automatically

In [5]:
array = [[1, 2, 3], 
         [4, 5, 6], 
         [7, 8, 9]]
print(array)
print(array[0][1])

#array[:,:] = 0

print(array)




[[1, 2, 3], [4, 5, 6], [7, 8, 9]]
2
[[1, 2, 3], [4, 5, 6], [7, 8, 9]]


In [6]:
T = [[11, 12, 5, 2], [11, 6,10], [10, 8, 12, 5], [1, 1]]
for r in T: #For each row in T is how that goes down
   for c in r: #For each element in each row
      print(c,end = " ") #print the c value
   print() #This makes a new line after each row

11 12 5 2 
11 6 10 
10 8 12 5 
1 1 


In [7]:
print("test")
print()
print("again")

test

again


In [8]:
for r in bandw: #For each row in T is how that goes down
    for c in r: #For each element in each row
        if c != 1:
            I1.text((3+col*grid_spacing+thk,row*grid_spacing+thk), str(num), font=myFont, fill=(0, 0, 0))
            num = num + 1
        col = col + 1 
    row = row + 1
    col = 0

In [3]:
len(bandw)
    

15